# Chapter 37: Data: regression and dimensionality reduction

The code for this chapter from [*Matrices in Practice*](https://trgoodale.github.io/matrix-book/part-6/37-data-science.html), by Tom Goodale.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trgoodale/matrix-book/blob/main/code/python/37-data-science.ipynb)

Run the cells in order, from the top: later cells use variables set in earlier ones. Change the numbers and run them again to see what happens. The code needs NumPy, SciPy and Matplotlib (and pandas in a few chapters), all of which Colab already has.

Code licensed under the [MIT licence](https://trgoodale.github.io/matrix-book/LICENSE.md).

In [ ]:
# The data files are in the book's repository. Use the local copy if there is one,
# otherwise read them from GitHub (as in Colab).
import os
DATA = "../../data/" if os.path.exists("../../data/iris.csv") else "https://raw.githubusercontent.com/trgoodale/matrix-book/main/data/"

### Code for a figure

This draws a figure in the chapter. You don't need it for the rest of the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.genfromtxt(DATA + "iris.csv", delimiter=",", skip_header=1, dtype=str)
X = data[:, :4].astype(float)
species = data[:, 4]
Z = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)
U, s, Vt = np.linalg.svd(Z, full_matrices=False)
scores = Z @ Vt[:2].T

fig, ax = plt.subplots(figsize=(6, 4))
for name, colour, marker in [("setosa", "#2153b8", "o"), ("versicolor", "#9c5208", "s"),
                             ("virginica", "#0a7050", "^")]:
    pts = scores[species == name]
    ax.scatter(pts[:, 0], pts[:, 1], s=18, color=colour, marker=marker, label=name, alpha=0.8)
ax.set_xlabel(f"PC1 ({100 * s[0]**2 / np.sum(s**2):.0f}% of variance)")
ax.set_ylabel(f"PC2 ({100 * s[1]**2 / np.sum(s**2):.0f}%)")
ax.legend(fontsize=9)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## In code

In [ ]:
import numpy as np

data = np.genfromtxt(DATA + "iris.csv", delimiter=",", skip_header=1, usecols=range(4))
sepal_len, sepal_wid, petal_len, petal_wid = data.T
n = len(sepal_len)

# Example 2: regression, with and without collinear variables
X1 = np.column_stack([np.ones(n), petal_wid])
X2 = np.column_stack([np.ones(n), petal_len, petal_wid])
for X in (X1, X2):
    beta, *_ = np.linalg.lstsq(X, sepal_len, rcond=None)
    resid = sepal_len - X @ beta
    R2 = 1 - resid @ resid / np.sum((sepal_len - sepal_len.mean())**2)
    print(np.round(beta, 4), f"R^2 = {R2:.3f}, cond = {np.linalg.cond(X):.1f}")

# Example 3: PCA of the standardised measurements
Z = (data - data.mean(axis=0)) / data.std(axis=0, ddof=1)
U, s, Vt = np.linalg.svd(Z, full_matrices=False)
print(np.round(s**2 / np.sum(s**2), 4))          # fraction of variance explained
print(np.round(Vt[0], 4))                        # loadings of the first component

## Exercises

The code exercises from the [end of the chapter](https://trgoodale.github.io/matrix-book/part-6/37-data-science.html#sec-data-science-exercises). Worked solutions are in the notebook `solutions-part-6.ipynb`, and in the [book's solutions](https://trgoodale.github.io/matrix-book/appendices/solutions-6.html#sec-solutions-data-science).

### Exercise 6

Fit sepal length on all three other measurements. Then refit using only the 50 *setosa* flowers. Compare the coefficients and their signs, and the condition numbers of the two design matrices. What does this suggest about fitting one model to data from several groups?

In [ ]:
# Your code here